# Socrata Hope and Pray Upload Script

This notebook is a proof of concept that we can replace datasync with python.  As such, it has been tested
to perform a replace on a test dataset.  This reads in a csv file, and runs the 6 checks/cleaning steps below.  Note that the Date Validation has yet to be tested.  This was successful in updating the dataset.
Testing now needs to be expanded to other data format, performing a append  (replace transfer from a post to a put), checking dates and boolean datatypes (Socrata Type Checkbox). I have set up a test dataset for testing ("rvak-43ap")

Column Check: Validates that the columns in the DataFrame match the expected columns from the type map, checking for missing, extra, and order-mismatched columns.
2. Numeric Cleaning: Cleans numeric columns by removing common formatting characters and converting them to numeric types, while tracking any errors and changes made.
3. Date Validation: Validates and optionally reformats date columns by attempting to parse them using a set of common date formats, while tracking any parsing errors.
4. Boolean Normalization: Cleans and normalizes boolean (checkbox) columns by converting common representations of true and false into consistent boolean values, while tracking any errors encountered during normalization.
5. String Length Check: Checks the lengths of string (text) columns and reports any values that exceed a specified maximum length (this stage is for reporting purposes only and does not modify the data).
6. JSON Serialization Check: Checks if the cleaned DataFrame can be successfully serialized to JSON format, which is important for downstream applications that require JSON data.


'''

In [2]:
import os
import sys

import requests
import json
import pandas as pd
import time
import numpy as np


# url = "https://data.colorado.gov/resource/rvak-43ap.json"



## Check Functions

In [5]:

def getMetadata(w4x4, username, password, api):
    '''
    Fetches metadata for a given Socrata dataset.

    Parameters:
    - w4x4 (str): The dataset identifier.
    - username (str): The Socrata username.
    - password (str): The Socrata password.
    - api (str): The Socrata API token.

    Returns:
    - dict: A dictionary mapping column names to their data types.
    '''
    columns={}
    meta = requests.get(
        f"https://data.colorado.gov/api/views/{w4x4}.json",
        headers={"X-App-Token": api},
        auth=(username, password),
    ).json()
    print(meta)
    cols = meta["columns"]

    for c in cols:
        columns[c["fieldName"]] = c["dataTypeName"]

    return columns 



def check_columns(df, type_map):
    '''
    Checks if the source data columns match the expected columns from the type map.  The Socrata 
    column names are case-insensitive, but the order must match exactly.  Extra or missing columns will be reported as errors, while order mismatches will be reported as warnings.

    Parameters:
    - df (pd.DataFrame): The DataFrame to check.
    - type_map (dict): A dictionary mapping column names to their expected data types.

    Returns:
    - dict: A dictionary containing information about missing, extra, and order-mismatched columns, as well as a validity flag.
    '''
    df_cols = list(df.columns.str.lower())
    socrata_cols = list(type_map.keys())

    extra = set(df_cols) - set(socrata_cols)
    missing = set(socrata_cols) - set(df_cols)

    if not extra and not missing:
        order_mismatch = df_cols != socrata_cols
    else:
        order_mismatch = False

    return {
        "missing_columns": sorted(missing),
        "extra_columns": sorted(extra),
        "order_mismatch": order_mismatch,
        "valid": (
            len(missing) == 0 and
            len(extra) == 0 and
            not order_mismatch
        )
    }


def clean_numeric_columns(df, type_map):
    '''Cleans numeric columns in the DataFrame based on the provided type map.  It identifies columns 
    that are expected to be numeric, removes common formatting characters 
    (commas, dollar signs, percent signs, parentheses), and attempts to convert them to 
    numeric types.  The function also tracks any errors encountered during conversion and any 
    changes made to the original data.
    
    
    Parameters:
    - df (pd.DataFrame): The DataFrame to clean.
    - type_map (dict): A dictionary mapping column names to their expected data types.  Only columns
      with types in the set {"number", "double", "money", "percent"} will be processed.
    
    Returns:
    - dict: A dictionary containing the cleaned DataFrame, any errors encountered, any changes made, and a validity flag.
    '''
    numeric_types = {"number", "double", "money", "percent"}

    df = df.copy()
    errors = {}
    changes = {}
    
    for col in df.columns:
        dtype = type_map.get(col.lower())
        if dtype not in numeric_types:
            continue

        original = df[col]

        # Clean string version
        cleaned = (
            original
            .astype(str)
            .str.replace(",", "", regex=False)
            .str.replace("$", "", regex=False)
            .str.replace("%", "", regex=False)
            .str.replace("(", "-", regex=False)
            .str.replace(")", "", regex=False)
            .str.strip()
        )

        # Convert to numeric
        converted = pd.to_numeric(cleaned, errors="coerce")

        # Missing logic
        is_missing = original.isna() | (cleaned == "")

        # Invalid values (fail)
        bad_mask = converted.isna() & (~is_missing)

        if bad_mask.any():
            errors[col] = {
                "bad_count": int(bad_mask.sum()),
                "examples": original.loc[bad_mask].head(5).tolist()
            }

        # --- CHANGE TRACKING ---
        original_str = original.astype(str)

        changed_mask = (original_str != cleaned) & (~is_missing)

        if changed_mask.any():
            changes[col] = {
                "count_changed": int(changed_mask.sum()),
                "examples_before": original.loc[changed_mask].head(3).tolist(),
                "examples_after": cleaned.loc[changed_mask].head(3).tolist()
            }
        else:
            changes[col] = {
                "count_changed": 0
            }

        # --- WRITE BACK ---
        df[col] = converted.astype(object)
        df.loc[converted.isna(), col] = ""

    return {
        "df": df,
        "errors": errors,
        "changes": changes,
        "valid": len(errors) == 0
    }



def validate_dates(
    df,
    type_map,
    date_formats=None,
    extra_date_formats=None,
    reformat=False
):
    '''Validates and optionally reformats date columns in the DataFrame based on the provided type map.  
    It identifies columns that are expected to be dates, attempts to parse them using a set of common date 
    formats (which can be extended with user-provided formats), and tracks any errors encountered during 
    parsing.  If reformatting is enabled and no parsing errors are found, it will reformat the dates into 
    a consistent format (YYYY-MM-DD for date-only values and YYYY-MM-DD HH:MM:SS for datetime values). 

    NOTE THIS HAS NOT BEEN TESTED AND HAS BEEN LEFT FOR THE NEXT PHASE.  THIS WILL REQUIRE DIFFERENT DATE
    FORMATS TO BE TESTED AND MAY REQUIRE ADJUSTMENTS TO THE LOGIC BASED ON THE RESULTS OF THOSE TESTS.
    ALSO MIGHT WANT TO CHECK SOME CUSTOM DATE FORMAT THAT CAN BE INPUT BY THE USER IN THE CONFIG TO HANDLE ANY UNIQUE FORMATS THAT ARE COMMON IN THE DATA.
 
    Parameters:
    - df (pd.DataFrame): The DataFrame to validate and reformat.
    - type_map (dict): A dictionary mapping column names to their expected data types.  Only columns with the type "calendar_date" will be processed.
    - date_formats (dict, optional): A dictionary mapping column names to lists of date formats to try for parsing.  If not provided, a default set of common date formats will be used for all date columns.
    - extra_date_formats (list, optional): A list of additional date formats to try for all date columns, in addition to the default formats or any column-specific formats provided in date_formats.
    - reformat (bool, optional): Whether to reformat successfully parsed dates into a consistent format.  If True, date-only values will be reformatted to "YYYY-MM-DD" and datetime values will be reformatted to "YYYY-MM-DD HH:MM:SS".  Reformatting will only be applied if no parsing errors are found.
    '''


    DEFAULT_DATE_FORMATS = [
        "%Y-%m-%d",
        "%m/%d/%Y",
        "%m/%d/%y",
        "%d-%b-%Y",
        "%m/%d/%Y %I:%M:%S %p",
        "%m/%d/%Y %I:%M:%S %p %Z",
        "%Y-%m-%d %H:%M:%S",
    ]

    df = df.copy()
    errors = {}

    for col in df.columns:
        dtype = type_map.get(col)

        if dtype != "calendar_date":
            continue

        if date_formats and col in date_formats:
            formats = list(date_formats[col])
        else:
            formats = DEFAULT_DATE_FORMATS.copy()
            if extra_date_formats:
                formats.extend(extra_date_formats)

        original = df[col]
        stripped = original.astype(str).str.strip()

        matched = pd.Series(False, index=df.index)
        parsed_result = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")

        for fmt in formats:
            parsed = pd.to_datetime(original, format=fmt, errors="coerce")
            success = parsed.notna() & (~matched)

            matched = matched | parsed.notna()
            parsed_result.loc[success] = parsed.loc[success]

        is_missing = original.isna() | (stripped == "")
        bad_mask = (~matched) & (~is_missing)

        if bad_mask.any():
            errors[col] = {
                "bad_count": int(bad_mask.sum()),
                "examples": original.loc[bad_mask].head(5).tolist(),
                "formats_tried": formats
            }

        if reformat and not bad_mask.any():
            df[col] = parsed_result.astype(object)
            df.loc[parsed_result.isna(), col] = ""

            has_time = (
                (parsed_result.dt.hour != 0) |
                (parsed_result.dt.minute != 0) |
                (parsed_result.dt.second != 0)
            )

            date_only_mask = parsed_result.notna() & (~has_time)
            datetime_mask = parsed_result.notna() & has_time

            df.loc[date_only_mask, col] = (
                parsed_result.loc[date_only_mask]
                .dt.strftime("%Y-%m-%d")
            )

            df.loc[datetime_mask, col] = (
                parsed_result.loc[datetime_mask]
                .dt.strftime("%Y-%m-%d %H:%M:%S")
            )

    return {
        "df": df,
        "errors": errors,
        "valid": len(errors) == 0
    }


def check_json_serializable(df):
    '''Checks if the DataFrame can be serialized to JSON format.  This is a final check to ensure that the cleaned DataFrame can be successfully converted to JSON, which is a common requirement for data interchange and storage.  The function attempts to convert the DataFrame to a list of records (dictionaries) and then serialize it to JSON.  If any errors occur during this process, they are caught and reported.  This check is important because even if the DataFrame passes all previous validation and cleaning steps, there may still be issues with certain data types or values that prevent successful JSON serialization.  By performing this check at the end of the pipeline, we can catch any remaining issues before the data is used for downstream applications. 
    Parameters:
    - df (pd.DataFrame): The DataFrame to check for JSON serializability.   
    Returns:
    - dict: A dictionary containing a validity flag and any error message encountered during JSON serialization.'''
    try:
        records = df.to_dict(orient="records")
        json.dumps(records)
        return {
            "valid": True,
            "error": None
        }
    except Exception as e:
        return {
            "valid": False,
            "error": str(e)
        }


def clean_boolean_columns(df, type_map, normalize=True):
    '''Cleans and normalizes boolean (checkbox) columns in the DataFrame based on the provided type map.  
    It identifies columns that are expected to be boolean (checkbox), and if normalization is enabled, it 
    attempts to convert various common representations of true and false values 
    (e.g., "yes", "no", "true", "false", "1", "0") into consistent boolean values (True, False) or empty s
    trings for missing values.  The function also tracks any errors encountered during normalization
    (values that cannot be interpreted as true, false, or missing) and reports examples of such values.  
    If normalization is disabled, the function will simply check for the presence of non-boolean values 
    and report them as errors without making any changes to the data.  

    Parameters:
    - df (pd.DataFrame): The DataFrame to clean.
    - type_map (dict): A dictionary mapping column names to their expected data types.  Only columns with the type "checkbox" will be processed.
    - normalize (bool, optional): Whether to normalize boolean values.  If True, the function will attempt to convert common representations of
      true and false into consistent boolean values (True, False) or empty strings for missing values.  If False, the function will only check for the presence of non-boolean values and report them as errors without making any changes to the data.
      Returns:
    - dict: A dictionary containing the cleaned DataFrame, any errors encountered, and a validity flag. Errors will include the count of bad values and examples of such values for each checkbox column that contains non-boolean values.  The cleaned DataFrame will have boolean columns normalized to True, False, or empty strings if normalization is enabled and no errors are found.    '''
    df = df.copy()
    errors = {}

    true_values = {"y", "yes", "true", "1"}
    false_values = {"n", "no", "false", "0"}

    for col in df.columns:
        dtype = type_map.get(col)

        if dtype != "checkbox":
            continue

        if not normalize:
            continue

        original = df[col]
        cleaned = original.astype(str).str.strip().str.lower()

        # Masks
        is_missing = original.isna() | (cleaned == "")
        is_true = cleaned.isin(true_values)
        is_false = cleaned.isin(false_values)

        # --- THE ACTUAL CHECK ---
        bad_mask = ~(is_true | is_false | is_missing)

        # --- THE ACTUAL REPLACEMENT ---
        df.loc[is_true, col] = True
        df.loc[is_false, col] = False
        df.loc[is_missing, col] = ""

        if bad_mask.any():
            errors[col] = {
                "bad_count": int(bad_mask.sum()),
                "examples": original.loc[bad_mask].head(5).tolist()
            }

    return {
        "df": df,
        "errors": errors,
        "valid": len(errors) == 0
    }


def check_string_lengths(df, type_map, max_len=10000):
    '''Checks the lengths of string (text) columns in the DataFrame based on the provided type map.
    It identifies columns that are expected to be text, calculates the length of each string value in those columns, and checks if any values exceed a specified maximum length.  The function tracks any issues found, including the count of values exceeding the maximum length, the maximum length found in the data, and examples of values that exceed the limit.  This check is important for ensuring that string data does not exceed expected limits, which can help prevent issues with data storage, processing, or display in downstream applications.  The function returns a report of any issues found without modifying the original DataFrame.
    Parameters:
    - df (pd.DataFrame): The DataFrame to check.
    - type_map (dict): A dictionary mapping column names to their expected data types.  Only columns with the type "text" will be processed.
    - max_len (int, optional): The maximum allowed length for string values.  Any values exceeding this length will be reported as issues.  The default value is 10,000 characters.
    Returns:
    - dict: A dictionary containing a report of any string length issues found, including the count of values exceeding the maximum length, the maximum length found in the data, and examples of values that exceed the limit for each text column.  The report also includes a flag indicating whether any issues were found.  The original DataFrame is not modified by this function. 
    '''
    issues = {}

    for col in df.columns:
        dtype = type_map.get(col)

        if dtype != "text":
            continue

        series = df[col].astype(str)
        lengths = series.str.len()

        long_mask = lengths > max_len

        if long_mask.any():
            issues[col] = {
                "count_exceeding": int(long_mask.sum()),
                "max_length_found": int(lengths.max()),
                "examples": series.loc[long_mask].head(3).tolist()
            }

    return {
        "issues": issues,
        "has_issues": len(issues) > 0
    }


def run_socrata_pipeline(
    df,
    type_map,
    date_formats=None,
    extra_date_formats=None,
    reformat_dates=True,
    normalize_booleans=True,
    max_string_length=10000
):
    '''Runs the full data validation and cleaning pipeline on a DataFrame based on the provided type map and configuration options.  The pipeline includes the following stages:
1. Column Check: Validates that the columns in the DataFrame match the expected columns from the type map, checking for missing, extra, and order-mismatched columns.
2. Numeric Cleaning: Cleans numeric columns by removing common formatting characters and converting them to numeric types, while tracking any errors and changes made.
3. Date Validation: Validates and optionally reformats date columns by attempting to parse them using a set of common date formats, while tracking any parsing errors.
4. Boolean Normalization: Cleans and normalizes boolean (checkbox) columns by converting common representations of true and false into consistent boolean values, while tracking any errors encountered during normalization.
5. String Length Check: Checks the lengths of string (text) columns and reports any values that exceed a specified maximum length (this stage is for reporting purposes only and does not modify the data).
6. JSON Serialization Check: Checks if the cleaned DataFrame can be successfully serialized to JSON format, which is important for downstream applications that require JSON data.
The function returns a comprehensive report of the results from each stage of the pipeline, including any errors encountered and changes made, as well as a final validity flag and the cleaned DataFrame if all stages are successful.  If any stage fails, the function will return immediately with the relevant error information without proceeding to subsequent stages.
Parameters:
- df (pd.DataFrame): The DataFrame to process through the pipeline.
- type_map (dict): A dictionary mapping column names to their expected data types, which guides the processing logic for each column in the various stages of the pipeline.
- date_formats (dict, optional): A dictionary mapping column names to lists of date formats to try for parsing date columns. If not provided, a default set of common date formats will be used for all date columns.
- extra_date_formats (list, optional): A list of additional date formats to try for all date columns, in addition to the default formats or any column-specific formats provided in date_formats.
- reformat_dates (bool, optional): Whether to reformat successfully parsed dates into a consistent format. If True, date-only values will be reformatted to "YYYY-MM-DD" and datetime values will be reformatted to "YYYY-MM-DD HH:MM:SS". Reformatting will only be applied if no parsing errors are found.
- normalize_booleans (bool, optional): Whether to normalize boolean (checkbox) columns by converting common representations of true and false into consistent boolean values (True, False) or empty strings for missing values. If False, the function will only check for the presence of non-boolean values and report them as errors without making any changes to the data.
- max_string_length (int, optional): The maximum allowed length for string values in text columns. Any values exceeding this length will be reported as issues in the string length check stage. The default value is 10,000 characters.
Returns:
- dict: A dictionary containing the results of each stage of the pipeline, including any errors encountered and changes made, as well as a final validity flag and the cleaned DataFrame if all stages are successful. If any stage fails, the function will return immediately with the relevant error information without proceeding to subsequent stages. The structure of the returned dictionary includes:
  - "valid": A boolean flag indicating whether the entire pipeline was successful (True if all stages passed, False if any stage failed).
  - "stage": A string indicating the stage at which the pipeline failed (e.g., "columns", "numeric", "dates", "boolean", "json") or "complete" if all stages were successful.
  - "results": A dictionary containing the results from each stage of the pipeline, including any errors and changes tracked.
  - "df": The cleaned DataFrame if the pipeline was successful, or None if any stage failed. 
'''
    results = {}

    # --- 1. Column Check ---
    col_check = check_columns(df, type_map)
    results["columns"] = col_check

    if not col_check["valid"]:
        return {
            "valid": False,
            "stage": "columns",
            "results": results,
            "df": None
        }

    # --- 2. Numeric Cleaning ---
    num_result = clean_numeric_columns(df, type_map)
    results["numeric"] = num_result
    
    if not num_result["valid"]:
        return {
            "valid": False,
            "stage": "numeric",
            "results": results,
            "df": None
        }

    df_work = num_result["df"]

    # --- 3. Date Validation ---
    date_result = validate_dates(
        df_work,
        type_map,
        date_formats=date_formats,
        extra_date_formats=extra_date_formats,
        reformat=reformat_dates
    )
    results["dates"] = date_result

    if not date_result["valid"]:
        return {
            "valid": False,
            "stage": "dates",
            "results": results,
            "df": None
        }

    df_work = date_result["df"]

    # --- 4. Boolean Normalization ---
    bool_result = clean_boolean_columns(
        df_work,
        type_map,
        normalize=normalize_booleans
    )
    results["boolean"] = bool_result

    if not bool_result["valid"]:
        return {
            "valid": False,
            "stage": "boolean",
            "results": results,
            "df": None
        }

    df_work = bool_result["df"]

    # --- 5. String Length Check (REPORT ONLY) ---
    length_result = check_string_lengths(
        df_work,
        type_map,
        max_len=max_string_length
    )
    results["string_length"] = length_result

    # --- 6. JSON Serialization Check ---
    json_result = check_json_serializable(df_work)
    results["json"] = json_result

    if not json_result["valid"]:
        return {
            "valid": False,
            "stage": "json",
            "results": results,
            "df": None
        }

    # --- SUCCESS ---
    return {
        "valid": True,
        "stage": "complete",
        "results": results,
        "df": df_work
    }

## Main

In [6]:
'''just a test run of the pipeline on the sample data to see the results of each stage and make sure
everything is working as expected before we integrate this into the full datasync process.  This will 
allow us to identify any issues or adjustments needed in the pipeline logic based on the actual data we
are working with.  We can also use this test run to verify that the error tracking and reporting is 
functioning correctly, and that the cleaned DataFrame is in the expected format for downstream use.  
Depending on the results of this test run, we may need to make adjustments to the type map, date formats,
or other configuration options to ensure that the pipeline is robust and effective for our specific 
dataset.      
'''

w4x4="rvak-43ap"
bic_home = os.getenv("bic_etl_home")

flog=open(f"{bic_home}/general/datasync/config.json")
info = json.load(flog)
username=info['username']
password=info['password']
api=info['appToken']



columns = getMetadata(w4x4, username, password, api)
df = pd.read_csv("Testing_for_Datasync_Change_20260415.csv")


## res is a dictionary containing the results of each stage of the pipeline, including any errors encountered and changes made, as well as a final validity flag and the cleaned DataFrame if all stages are successful.  If any stage fails, the function will return immediately with the relevant error information without proceeding to subsequent stages. The structure of the returned dictionary includes:
# - "valid": A boolean flag indicating whether the entire pipeline was successful (True if all stages passed, False if any stage failed).
# - "stage": A string indicating the stage at which the pipeline failed (e.g., "columns", "numeric", "dates", "boolean", "json") or "complete" if all stages were successful.
# - "results": A dictionary containing the results from each stage of the pipeline, including any errors and changes tracked.
# - "df": The cleaned DataFrame if the pipeline was successful, or None if any stage failed.

res = run_socrata_pipeline(
    df,
    columns
)

max_retries = 5
base_url = "https://data.colorado.gov"



df=res["df"]

df = df.fillna("")
data = df.to_dict(orient="records")
#dataset_id = "rvak-43ap"

url = f"{base_url}/resource/{w4x4}.json"

for attempt in range(max_retries):
    response = requests.put(
        url,
        headers={"X-App-Token": api},
        auth=(username, password),
        json=data
    )

    if response.status_code == 200:
            print(f"✅ Replace successful ({len(data)} rows)")
            break
      

    if attempt < max_retries - 1:
        
        time.sleep(5)





{'code': 'not_found', 'error': True, 'message': 'Not found', 'data': {'id': 'rvak-43ap'}}


KeyError: 'columns'